# 01 Cechy – próba tygodniowa

## 1. Konfiguracja

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.config import MAX_PER_WEEK, MIN_WORDS_SAMPLE, PROCESSED_DIR, RANDOM_STATE, STUDY_FILES

CLEAN_DIR = PROCESSED_DIR / "clean"
columns = ["id", "link_id", "subreddit", "camp", "cycle", "week_start", "phase", "text", "n_words", "keep"]

## 2. Pula komentarzy

In [2]:
pool = []
for subreddit, cycle in STUDY_FILES:
    df = pd.read_parquet(CLEAN_DIR / f"r_{subreddit}_comments_{cycle}.parquet", columns=columns)
    pool.append(df[df["keep"] & (df["n_words"] >= MIN_WORDS_SAMPLE)].drop(columns="keep"))
pool = pd.concat(pool, ignore_index=True)
pool.shape

(13465220, 9)

## 3. Losowanie do `MAX_PER_WEEK` na subreddit i tydzień

In [3]:
sample = pool.sample(frac=1, random_state=RANDOM_STATE).groupby(["subreddit", "week_start"]).head(MAX_PER_WEEK)
sample = sample.sort_values(["subreddit", "cycle", "week_start"]).reset_index(drop=True)
sample.to_parquet(PROCESSED_DIR / "sample.parquet", index=False)
sample.shape

(794519, 9)

## 4. Liczebność próby

In [4]:
sizes = sample.groupby(["subreddit", "cycle", "phase"], dropna=False).size().unstack("phase")
sizes

phase                   bazowa  powyborcza  przedwyborcza  przejsciowa  \
subreddit        cycle                                                   
Conservative     2012   3400.0      2400.0        19200.0       3300.0   
                 2016   5700.0      2400.0        15000.0       3300.0   
                 2020   4200.0      2400.0        18300.0       3300.0   
                 2024   6600.0      2400.0        15600.0       3300.0   
Liberal          2012    350.0      2400.0         9025.0       3032.0   
                 2016   5611.0      2320.0        14885.0       3300.0   
                 2020   4021.0      2400.0        17021.0       3182.0   
                 2024   6595.0      2400.0        15410.0       3300.0   
Republican       2012    727.0      2400.0        14875.0       3300.0   
                 2016   5091.0      2400.0        15000.0       3300.0   
                 2020   4200.0      2400.0        18000.0       3241.0   
                 2024   6600.0      2400.0        15600.0       3300.0   
democrats        2012    375.0      1717.0         5769.0       1848.0   
                 2016   5045.0      2400.0        14656.0       3300.0   
                 2020   4200.0      2400.0        18300.0       3300.0   
                 2024   6600.0      2400.0        15600.0       3300.0   
languagelearning 2012   1617.0      2376.0        14607.0       3300.0   
                 2014      NaN         NaN            NaN          NaN   
                 2016   5700.0      2400.0        15000.0       3300.0   
                 2018      NaN         NaN            NaN          NaN   
                 2020   4200.0      2400.0        18300.0       3300.0   
                 2022      NaN         NaN            NaN          NaN   
                 2024   6600.0      2400.0        15600.0       3300.0   
running          2012   3600.0      2400.0        19200.0       3300.0   
                 2014      NaN         NaN            NaN          NaN   
                 2016   5700.0      2400.0        15000.0       3300.0   
                 2018      NaN         NaN            NaN          NaN   
                 2020   4200.0      2400.0        18300.0       3300.0   
                 2022      NaN         NaN            NaN          NaN   
                 2024   6600.0      2400.0        15600.0       3300.0   

phase                   rok_referencyjny  szczytowa    NaN  
subreddit        cycle                                      
Conservative     2012                NaN     2700.0    NaN  
                 2016                NaN     4800.0  300.0  
                 2020                NaN     3000.0  300.0  
                 2024                NaN     3300.0    NaN  
Liberal          2012                NaN     2044.0    NaN  
                 2016                NaN     4062.0  300.0  
                 2020                NaN     3000.0  300.0  
                 2024                NaN     3162.0    NaN  
Republican       2012                NaN     2700.0    NaN  
                 2016                NaN     4800.0  300.0  
                 2020                NaN     2987.0  300.0  
                 2024                NaN     3300.0    NaN  
democrats        2012                NaN     1810.0    NaN  
                 2016                NaN     4584.0  300.0  
                 2020                NaN     3000.0  300.0  
                 2024                NaN     3300.0    NaN  
languagelearning 2012                NaN     2574.0    NaN  
                 2014            15600.0        NaN  300.0  
                 2016                NaN     4800.0  300.0  
                 2018            15600.0        NaN  300.0  
                 2020                NaN     3000.0  300.0  
                 2022            15600.0        NaN  300.0  
                 2024                NaN     3300.0    NaN  
running          2012                NaN     2700.0    NaN  
                 2014            15600.0     

In [5]:
weekly_sizes = sample.groupby(["subreddit", "cycle", "week_start"]).size()
(weekly_sizes < MAX_PER_WEEK).groupby(["subreddit", "cycle"]).sum().unstack("cycle")

cycle,2012,2014,2016,2018,2020,2022,2024
subreddit,,,,,,,
Conservative,6.0,NaN,0.0,NaN,0.0,NaN,0.0
Liberal,79.0,NaN,16.0,NaN,22.0,NaN,6.0
Republican,43.0,NaN,10.0,NaN,2.0,NaN,0.0
democrats,101.0,NaN,23.0,NaN,0.0,NaN,0.0
languagelearning,56.0,0.0,0.0,0.0,0.0,0.0,0.0
running,0.0,0.0,0.0,0.0,0.0,0.0,0.0
